# 06 · Probabilistic Forecasting — Quantiles & Prediction Intervals

The most valuable output of TimesFM is **uncertainty**. A point forecast says
"we'll sell 100". A probabilistic forecast says "80% chance between 82 and 121".
This notebook shows how to read and use every quantile.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
rng = np.random.default_rng(3)
t = np.arange(240)
series = (100 + 20*np.sin(2*np.pi*t/24) + rng.normal(0, 6, t.size)).astype(np.float32)
point, q = model.forecast(horizon=24, inputs=[series])
point, q = point[0], q[0]
print("quantile array shape:", q.shape, "(horizon, 10)")

## Understanding the output

`model.forecast()` returns **two** arrays:

| Array | Shape | Meaning |
| ----- | ----- | ------- |
| `point_forecast` | `(n_series, horizon)` | the median (0.5 quantile) point forecast |
| `quantile_forecast` | `(n_series, horizon, 10)` | probabilistic bands |

The last axis of `quantile_forecast` has **10 slices**:

| Index | Quantile | Use |
| ----- | -------- | --- |
| `0` | mean | average prediction (NOT q0!) |
| `1` | 0.1 | lower bound of the 80% interval |
| `2` | 0.2 | lower bound of the 60% interval |
| `5` | 0.5 | median (== `point_forecast`) |
| `8` | 0.8 | upper bound of the 60% interval |
| `9` | 0.9 | upper bound of the 80% interval |

> ⚠️ **Common mistake:** index `0` is the **mean**, not the 0th percentile.
> q10 lives at index `1`, q90 at index `9`.

In [ ]:
# Every quantile at the first forecast step
labels = ["mean", "q10", "q20", "q30", "q40", "q50", "q60", "q70", "q80", "q90"]
for j, lab in enumerate(labels):
    print(f"step 1 {lab:>4}: {q[0, j]:8.2f}")

## Fan chart — visualise the full predictive distribution

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

xf = range(24)
fig, ax = plt.subplots(figsize=(12, 5))
# nested bands from widest to narrowest
pairs = [(1, 9, 0.12, "80%"), (2, 8, 0.18, "60%"), (3, 7, 0.25, "40%"), (4, 6, 0.35, "20%")]
for lo, hi, a, lab in pairs:
    ax.fill_between(xf, q[:, lo], q[:, hi], color="tab:purple", alpha=a, label=lab)
ax.plot(xf, q[:, 5], color="black", lw=2, label="median")
ax.set_title("Predictive fan chart"); ax.legend(ncol=5, fontsize=8)
fig.tight_layout(); fig.savefig("fan_chart.png", dpi=130)
print("saved fan_chart.png")

## Turning quantiles into business decisions

- **Safety stock / service level:** order up to `q90` to cover 90% of demand
  scenarios. Higher service level → higher quantile.
- **Risk budgeting:** treat `q10` as your conservative "worst realistic" case.
- **Interval width** (`q90 - q10`) is a direct measure of forecast *confidence*
  — flag steps where it explodes.

In [ ]:
service_level_90 = q[:, 9]     # cover 90% of scenarios
conservative     = q[:, 1]     # 10th percentile
interval_width   = q[:, 9] - q[:, 1]
print("order-up-to (q90), first 6 :", service_level_90[:6].round(1))
print("interval width, first 6     :", interval_width[:6].round(1))